# GenAI-Traces: Token Counting & Cost Estimation

This notebook demonstrates:
- Token counting with tiktoken
- Cost estimation for various models
- Tracking costs across traces

In [ ]:
import sys
sys.path.insert(0, '..')

## 1. Token Counting

In [ ]:
from genai_traces.telemetry.tokens.counter import TokenCounter, count_tokens, count_message_tokens

counter = TokenCounter()

# Count tokens in a simple string
text = "Hello, how are you doing today? I hope you're having a great day!"
tokens = counter.count(text, model="gpt-4o")
print(f"Text: {text}")
print(f"Token count (gpt-4o): {tokens}")

In [ ]:
# Count tokens for chat messages
messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "What is the capital of France?"},
    {"role": "assistant", "content": "The capital of France is Paris."},
    {"role": "user", "content": "What about Germany?"},
]

message_tokens = counter.count_messages(messages, model="gpt-4o")
print(f"\nChat messages token count: {message_tokens}")

# Compare with different models
for model in ["gpt-4o", "gpt-4o-mini", "gpt-3.5-turbo"]:
    tokens = counter.count_messages(messages, model=model)
    print(f"  {model}: {tokens} tokens")

## 2. Cost Estimation

In [ ]:
from genai_traces.telemetry.cost.estimator import CostEstimator, estimate_cost

estimator = CostEstimator()

# Estimate cost for a typical API call
costs = estimator.estimate(
    model="gpt-4o",
    prompt_tokens=1000,
    completion_tokens=500,
)

print("Cost breakdown for gpt-4o (1000 input, 500 output tokens):")
print(f"  Input cost:  ${costs['input_cost_usd']:.6f}")
print(f"  Output cost: ${costs['output_cost_usd']:.6f}")
print(f"  Total cost:  ${costs['total_cost_usd']:.6f}")

In [ ]:
# Compare costs across models
models = ["gpt-4o", "gpt-4o-mini", "gpt-4-turbo", "gpt-3.5-turbo", "claude-3-5-sonnet-20241022"]

print("\nCost comparison (1000 input, 500 output tokens):")
print("-" * 50)

for model in models:
    costs = estimator.estimate(model, 1000, 500)
    print(f"{model:35} ${costs['total_cost_usd']:.6f}")

In [ ]:
# Cost with cached tokens (Anthropic/OpenAI prompt caching)
costs_with_cache = estimator.estimate(
    model="gpt-4o",
    prompt_tokens=1000,
    completion_tokens=500,
    cached_tokens=800,  # 800 of the 1000 input tokens were cached
)

print("\nCost with prompt caching (800 cached tokens):")
print(f"  Input cost:  ${costs_with_cache['input_cost_usd']:.6f}")
print(f"  Cache cost:  ${costs_with_cache['cache_cost_usd']:.6f}")
print(f"  Output cost: ${costs_with_cache['output_cost_usd']:.6f}")
print(f"  Total cost:  ${costs_with_cache['total_cost_usd']:.6f}")
print(f"  Savings:     ${costs['total_cost_usd'] - costs_with_cache['total_cost_usd']:.6f}")

## 3. Integrated Tracing with Cost Tracking

In [ ]:
from genai_traces import init_tracer
from genai_traces.exporters import ConsoleExporter
from genai_traces.core.types import SpanType

tracer = init_tracer(
    service_name="cost-tracking-demo",
    exporters=[ConsoleExporter(pretty=True)],
)

# Simulate an LLM call with cost tracking
with tracer.start_as_current_span("llm_with_cost", SpanType.LLM) as span:
    # Set model info
    model = "gpt-4o"
    span.set_attribute("llm.model.name", model)
    span.set_attribute("llm.provider", "openai")
    
    # Simulate prompt and completion
    prompt = "Explain quantum computing in simple terms."
    completion = "Quantum computing uses quantum bits (qubits) that can exist in multiple states simultaneously..."
    
    span.set_attribute("llm.prompt", prompt)
    span.set_attribute("llm.completion", completion)
    
    # Count tokens
    prompt_tokens = counter.count(prompt, model)
    completion_tokens = counter.count(completion, model)
    
    span.set_attribute("llm.prompt_tokens", prompt_tokens)
    span.set_attribute("llm.completion_tokens", completion_tokens)
    span.set_attribute("llm.total_tokens", prompt_tokens + completion_tokens)
    
    # Calculate and record cost
    costs = estimator.estimate(model, prompt_tokens, completion_tokens)
    span.set_attribute("cost.input_cost_usd", costs['input_cost_usd'])
    span.set_attribute("cost.output_cost_usd", costs['output_cost_usd'])
    span.set_attribute("cost.total_usd", costs['total_cost_usd'])

print("\nSpan with cost tracking completed!")

## Summary

This notebook demonstrated:
- ✅ Token counting with tiktoken
- ✅ Message token counting with overhead
- ✅ Cost estimation for various models
- ✅ Prompt caching cost savings
- ✅ Integrated cost tracking in traces